In [2]:
# Initialize Otter
import otter
grader = otter.Notebook("practice01_full_solution.ipynb")

# ASSIGNMENT CONFIG
solutions_pdf: true
export_cell:
    instructions: "Submit the .zip file to our Moodle assignment page."
generate: 
    pdf: true
    filtering: true
    pagebreaks: true
    zips: false

**Student names and e-mails:**

_Test Student — test.student@calvin.edu_


# Practice 01 — DataFrame Basics

In this practice you will work with a real dataset about forest fires in Brazil. Each task is tagged with the SLO it covers:

| SLO | Description |
|-----|-------------|
| **02A** | Manipulate the structure and contents of pandas DataFrames |
| **02B** | Sort, filter, and query DataFrames |
| **02C** | Choose appropriate visual encodings in a visualization |

---
## The Dataset: Amazon Forest Fires (Brazil, 1998–2017)

![Amazon forest fire](https://images.unsplash.com/photo-1511027643875-5cbb0439c8f1?q=80&w=1200&auto=format&fit=crop)

This dataset reports monthly counts of forest fires by state, recorded by Brazil's national space research agency INPE. Each row represents one state–month–year combination.

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

The cell below loads the data from a CSV file into a pandas DataFrame called `fires`. Run it and look at the first few rows.

In [4]:
fires = pd.read_csv(
    'https://cs.calvin.edu/courses/data/202/26fa/datasets/amazon.csv',
    encoding='latin-1'
)
fires.head()

,year,state,month,number,date
0,1998,Acre,Janeiro,0.0,1998-01-01
1,1999,Acre,Janeiro,0.0,1999-01-01
2,2000,Acre,Janeiro,0.0,2000-01-01
3,2001,Acre,Janeiro,0.0,2001-01-01
4,2002,Acre,Janeiro,0.0,2002-01-01


Notice that the `month` column contains **Portuguese names** (e.g., `Janeiro`, `Fevereiro`). The cell below translates them to integers (1–12) and corrects the `date` column to match. Run it — you don't need to modify it.

In [5]:
month_map = {
    'Janeiro': 1, 'Fevereiro': 2, 'Marco': 3, 'Abril': 4,
    'Maio': 5, 'Junho': 6, 'Julho': 7, 'Agosto': 8,
    'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12
}
fires['month'] = fires['month'].map(month_map)
fires['date'] = pd.to_datetime(fires[['year', 'month']].assign(day=1))
fires.head()

,year,state,month,number,date
0,1998,Acre,1.0,0.0,1998-01-01
1,1999,Acre,1.0,0.0,1999-01-01
2,2000,Acre,1.0,0.0,2000-01-01
3,2001,Acre,1.0,0.0,2001-01-01
4,2002,Acre,1.0,0.0,2002-01-01


---
## Part 1 — Understanding the DataFrame

Before any analysis, explore what you have. Key tools:

| Expression | What it gives you |
|---|---|
| `df.shape` | Tuple `(rows, columns)` |
| `df.columns` | Index of column names |
| `df['col']` | A Series (one column) |
| `df['col'].sum()` | Total of that column |
| `df['col'].idxmax()` | Index of the max value |
| `df.loc[idx, 'col']` | Value at row `idx`, column `'col'` |

In [6]:
fires.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6454 entries, 0 to 6453
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   year    6454 non-null   int64         
 1   state   6454 non-null   object        
 2   month   5914 non-null   float64       
 3   number  6454 non-null   float64       
 4   date    5914 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 252.2+ KB


In [ ]:
fires.describe()

### Task 02A.1 — Shape and Columns *(1 pt)*

Using `fires.shape`, assign:
- `n_rows` to the number of rows
- `n_cols` to the number of columns

Use the attribute — don't type the literal numbers.

In [7]:
n_rows = fires.shape[0]
n_cols = fires.shape[1]
print(f'Rows: {n_rows}  |  Columns: {n_cols}')

Rows: 6454  |  Columns: 5


In [8]:
grader.check("02A.1")

02A.1 results: All test cases passed!

### Task 02A.2 — Accessing Columns and Computing *(2 pts)*

Using the `number` column (fires per record) and the `state` column:

1. Assign the **total** number of fires across all rows to `total_fires`.
2. Find the **state that had the single highest monthly fire count** in the dataset. Assign its name (a string) to `peak_state`.

*Hints: `.sum()`, `.idxmax()`, and `.loc[]` will be useful.*

In [9]:
total_fires = fires['number'].sum()
peak_idx = fires['number'].idxmax()
peak_state = fires.loc[peak_idx, 'state']
print(f'Total fires across all records: {total_fires:,.0f}')
print(f'State with highest single-month count: {peak_state}')

Total fires across all records: 698,924
State with highest single-month count: Amazonas


In [10]:
grader.check("02A.2")

02A.2 results: All test cases passed!

### Task 02A.3 — Adding a Column *(2 pts)*

Add a new column called `'decade'` to `fires` that records the **decade** of each row as a string:

- 1990–1999 → `'1990s'`
- 2000–2009 → `'2000s'`
- 2010–2019 → `'2010s'`

*Hint: integer division (`//`) on the `year` column gets you the decade start; string concatenation adds the `'s'`.*

In [11]:
fires['decade'] = (fires['year'] // 10 * 10).astype(str) + 's'
fires[['year', 'decade']].drop_duplicates().sort_values('year').head(10)

,year,decade
0,1998,1990s
1,1999,1990s
2,2000,2000s
3,2001,2000s
4,2002,2000s
5,2003,2000s
6,2004,2000s
7,2005,2000s
8,2006,2000s
9,2007,2000s


In [12]:
grader.check("02A.3")

02A.3 results: All test cases passed!

---
## Part 2 — Sorting and Filtering

Pandas lets you focus on subsets of your data:

| Operation | Syntax | Example |
|---|---|---|
| Boolean filter | `df[condition]` | `df[df['year'] > 2010]` |
| Multiple conditions | `df[cond1 & cond2]` | Use `&` not `and` |
| Filter by list | `df['col'].isin(lst)` | checks membership |
| Sort | `df.sort_values('col', ascending=False)` | largest first |
| Top N | `.head(N)` after sort | |

In [13]:
# Records from 2015 onward, during August (peak fire season)
late_august = fires[(fires['year'] >= 2015) & (fires['month'] == 8)]
print(f'{len(late_august)} rows matched')
late_august.sort_values('number', ascending=False).head()

81 rows matched


,year,state,month,number,date,decade
3745,2017,Paraiba,8.0,987.0,2017-08-01,2010s
5896,2017,Sao Paulo,8.0,981.0,2017-08-01,2010s
2070,2015,Goias,8.0,943.0,2015-08-01,2010s
1114,2015,Bahia,8.0,829.0,2015-08-01,2010s
4940,2017,Rio,8.0,819.0,2017-08-01,2010s


### Task 02B.1 — Filtering *(2 pts)*

Filter `fires` to include only rows for the state of **`'Mato Grosso'`**. Assign the result to `mato_grosso`.

Then print the number of rows in your filtered dataframe.

In [16]:
mato_grosso = fires[fires['state'] == 'Mato Grosso']
print(f'Mato Grosso rows: {len(mato_grosso)}')
mato_grosso.head()

Mato Grosso rows: 478


,year,state,month,number,date,decade
2391,1998,Mato Grosso,1.0,0.0,1998-01-01,1990s
2392,1999,Mato Grosso,1.0,39.0,1999-01-01,1990s
2393,2000,Mato Grosso,1.0,44.0,2000-01-01,2000s
2394,2001,Mato Grosso,1.0,44.0,2001-01-01,2000s
2395,2002,Mato Grosso,1.0,172.0,2002-01-01,2000s


In [17]:
grader.check("02B.1")

02B.1 results: All test cases passed!

### Task 02B.2 — Sorting *(2 pts)*

From the full `fires` dataframe, find the **5 rows with the highest fire counts**. Assign the result to `top5`.

Then display only the `state`, `year`, `month`, and `number` columns of `top5`.

In [18]:
top5 = fires.sort_values('number', ascending=False).head(5)
top5[['state', 'year', 'month', 'number']]

,state,year,month,number
888,Amazonas,2008,9.0,998.0
1105,Bahia,2006,8.0,995.0
1410,Ceara,2012,11.0,995.0
6346,Tocantins,2009,7.0,989.0
3745,Paraiba,2017,8.0,987.0


In [19]:
grader.check("02B.2")

02B.2 results: All test cases passed!

### Task 02B.3 — Compound Filtering *(2 pts)*

The north region of Brazil (the Amazon basin) includes these states:

```python
north_states = ['Acre', 'Amapa', 'Amazonas', 'Para', 'Rondonia', 'Roraima', 'Tocantins']
```

Create a new dataframe `north_recent` containing only rows where **both** conditions hold:
1. The state is in `north_states`
2. The year is **2010 or later**

Print its shape.

In [20]:
north_states = ['Acre', 'Amapa', 'Amazonas', 'Para', 'Rondonia', 'Roraima', 'Tocantins']
north_recent = fires[fires['state'].isin(north_states) & (fires['year'] >= 2010)]
print(f'north_recent shape: {north_recent.shape}')
north_recent.head()

north_recent shape: (570, 6)


,year,state,month,number,date,decade
12,2010,Acre,1.0,1.0,2010-01-01,2010s
13,2011,Acre,1.0,0.0,2011-01-01,2010s
14,2012,Acre,1.0,0.0,2012-01-01,2010s
15,2013,Acre,1.0,0.0,2013-01-01,2010s
16,2014,Acre,1.0,0.0,2014-01-01,2010s


In [21]:
grader.check("02B.3")

02B.3 results: All test cases passed!

---
## Part 3 — Visual Encodings

A chart maps data to **visual properties** called **encodings**:

| Encoding | Plotly Express argument | Best for |
|---|---|---|
| x-axis | `x=` | time, ordered categories |
| y-axis | `y=` | numeric values |
| color | `color=` | categories or continuous gradient |
| size | `size=` | quantity (use carefully) |
| facet | `facet_col=` | small multiples by category |

In [22]:
fires_per_year = fires.groupby('year')['number'].sum().reset_index()

fig_example = px.bar(
    fires_per_year,
    x='year',
    y='number',
    title='Total Forest Fires in Brazil per Year',
    labels={'number': 'Number of Fires', 'year': 'Year'}
)
fig_example.show()

### Task 02C.1 — Line Plot for One State *(2 pts)*

Using your `mato_grosso` dataframe from Task 02B.1, create a **line plot** showing fire counts over time:

- x-axis: `'date'`
- y-axis: `'number'`
- A descriptive title
- Axis labels via the `labels=` argument

Assign the figure to `fig1` and display it.

In [23]:
fig1 = px.line(
    mato_grosso,
    x='date',
    y='number',
    title='Forest Fires in Mato Grosso Over Time',
    labels={'number': 'Number of Fires', 'date': 'Date'}
)
fig1.show()

In [24]:
grader.check("02C.1")

02C.1 results: All test cases passed!

### Task 02C.2 — Comparing Multiple States *(3 pts)*

One state is interesting, but comparison reveals the bigger picture. Using your `north_recent` dataframe (Task 02B.3), create a line plot comparing fire trends across all north region states.

Requirements:
- Chart type: **line**
- x-axis: `'date'`
- y-axis: `'number'`
- **Color** encoding: `'state'`
- A title and axis labels

Assign to `fig2` and display it.

In [25]:
fig2 = px.line(
    north_recent,
    x='date',
    y='number',
    color='state',
    title='Forest Fires in Northern Brazil (2010–2017)',
    labels={'number': 'Number of Fires', 'date': 'Date', 'state': 'State'}
)
fig2.show()

In [26]:
grader.check("02C.2")

02C.2 results: All test cases passed!

<!-- BEGIN QUESTION -->

### Task 02C.3 — Evaluate Your Visualization *(2 pts)*

Look critically at `fig2`. In **3–5 sentences**, answer:

1. Does the color encoding help you compare states? Why or why not?
2. What would you change to make the patterns clearer?
3. What does this plot **not** show — what information is hidden or lost?

*Edit the cell below and write your answer.*

The color encoding makes it possible to compare the seven northern states at a glance, but with this many overlapping lines it gets hard to track any single state, especially where values are close together or lines cross. A clearer version could use small multiples (`facet_col='state'`) instead of overlaying every line in one color palette, or could highlight only the one or two most notable states while graying out the rest. The plot also hides the *why* behind each spike — it shows fire counts over time but nothing about rainfall, deforestation policy, or land use changes that likely explain the trends we see.

<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)